# Problem set 2: Descriptive economics

**Learning goals:**

- Using basic `pandas` DataFrame operations
- Clean and structure data
- Download real economic datasets using an API
- Merge and join data sets
- Compute summary statistics

**Table of contents**<a id='toc0_'></a>    
- 1. [Basic pandas](#toc1_)    
  - 1.1. [Dataframe](#toc1_1_)    
  - 1.2. [New variable](#toc1_2_)    
  - 1.3. [Indexing](#toc1_3_)    
  - 1.4. [Changing variables](#toc1_4_)    
  - 1.5. [Dropping observations and columns](#toc1_5_)    
  - 1.6. [Renaming](#toc1_6_)    
  - 1.7. [Income distribution](#toc1_7_)    
- 2. [National account identity](#toc2_)    
  - 2.1. [Download](#toc2_1_)    
  - 2.2. [Merge](#toc2_2_)    
  - 2.3. [Split-apply-combine-plot](#toc2_3_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

**Tips:**

**Copy** a DataFrame to ensure you do not modify the original data.

**Masks** are used to filter data in a DataFrame based on specific conditions. A mask is essentially a boolean array that indicates whether each element in the DataFrame meets the condition (True) or not (False). When you apply this mask to the DataFrame, it returns only the rows where the condition is True.

**loc** and **iloc** are used to access specific rows and columns in a DataFrame. **loc** is label-based, meaning you use the actual labels of the rows and columns to select data. **iloc** is integer position-based, meaning you use the integer index positions to select data.

**Masks** work with both **loc** and **iloc**, but they are typically used with **loc** because masks are usually based on the actual data values (labels) rather than their integer positions.

Modules

In [31]:
import numpy as np
import pandas as pd
from IPython.display import display

import matplotlib.pyplot as plt
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

# install with: !pip install "git+https://github.com/alemartinello/dstapi`
# !pip install "git+https://github.com/alemartinello/dstapi"

from dstapi import DstApi 

## 1. <a id='toc1_'></a>[Basic pandas](#toc0_)

### 1.1. <a id='toc1_1_'></a>[Dataframe](#toc0_)

Modify the code below such that *income* and *consumption* are variables in the *df* DataFrame.

In [32]:
# Jeppe sets a random seed for reproducibility
np.random.seed(1999)
 
# And creates some fake data
N = 100
mydata = {}
mydata['id'] = range(N)
income = np.exp(np.random.normal(size=N)) # Picks the random pseudo-random numbers the same way each time due to the seed
consumption = np.sqrt(income)

# We'll add income and consumption to the data dictionary
mydata['income'] = income
mydata['consumption'] = consumption

# We create a pandas DataFrame
df = pd.DataFrame(mydata)

# Display the first five rows
df.head() # Display the first five rows

,id,income,consumption
0,0,0.727981,0.853218
1,1,1.997831,1.413447
2,2,0.276823,0.526140
3,3,1.481931,1.217346
4,4,1.235904,1.111712


Important! Create a copy of the DataFrame called *df_true* before you start modifying it. We will use the original DataFrame later.

In [33]:
# Create a copy of the original dataframe to not overwrite it
df_true = df.copy()

In [34]:
# Display the whole dataframe
display(df_true)

,id,income,consumption
0,0,0.727981,0.853218
1,1,1.997831,1.413447
2,2,0.276823,0.526140
3,3,1.481931,1.217346
4,4,1.235904,1.111712
...,...,...,...
95,95,0.201856,0.449284
96,96,2.368034,1.538842
97,97,2.389874,1.545922
98,98,1.488635,1.220096


### 1.2. <a id='toc1_2_'></a>[New variable](#toc0_)

Create a new variable *ratio* which is the ratio of consumption to income.

In [35]:
# We create the new variable and add it to the dataframe in one go by defining a new column
df_true["ratio"] = df["consumption"] / df["income"]

# Show the first five rows again to see the new variable
df_true.head()

,id,income,consumption,ratio
0,0,0.727981,0.853218,1.172033
1,1,1.997831,1.413447,0.707490
2,2,0.276823,0.526140,1.900636
3,3,1.481931,1.217346,0.821459
4,4,1.235904,1.111712,0.899513


### 1.3. <a id='toc1_3_'></a>[Indexing](#toc0_)

**Question a:** Select everybody: with an income above 1.

In [36]:
# We create a mask for income > 1
mask = df_true["income"] > 1

# We use the mask to filter the dataframe and display the result using head
df_true.loc[mask, :].head()

,id,income,consumption,ratio
1,1,1.997831,1.413447,0.707490
3,3,1.481931,1.217346,0.821459
4,4,1.235904,1.111712,0.899513
6,6,2.574032,1.604379,0.623294
7,7,2.475478,1.573365,0.635580


**Question b:** Select everybody with an income *above* 1 and a ratio *above* 0.7.

In [37]:
# We create another mask for income above 1 and ratio above 0.7
mask = ((df_true["income"] > 1) & (df_true["ratio"] > 0.7))

# We use the mask to filter the dataframe and display the result using head
df_true.loc[mask, :].head()

,id,income,consumption,ratio
1,1,1.997831,1.413447,0.707490
3,3,1.481931,1.217346,0.821459
4,4,1.235904,1.111712,0.899513
11,11,2.031708,1.425380,0.701567
18,18,1.280235,1.131475,0.883802


### 1.4. <a id='toc1_4_'></a>[Changing variables](#toc0_)

**Question a:** Set consumption equal to 0.5 if income is less than 0.5.

In [38]:
# We create a mask for income < 0.5
mask = df_true["income"] < 0.5

# We use the mask to set consumption to 0.5 for all rows where the mask is true
df_true.loc[mask, "consumption"] = 0.5 # Interpretation: For all rows in the column "consumption" where the mask is true, set the value to 0.5

# Print the mean of consumption with and without mask
print("Mean of consumption with mask: ", df_true['consumption'].mean()) # <- With mask
print("Mean of consumption without mask: ", df['consumption'].mean()) # <- Without mask

Mean of consumption with mask:  1.075479712048503
Mean of consumption without mask:  1.0878444061731818


**Question b:** Set consumption equal to income if income is less than 0.5.

In [39]:
# We create a mask for income < 0.5
mask = df_true["income"] < 0.5

# We use the mask to set consumption to income if the mask is true
df_true.loc[mask, "consumption"] = df_true.loc[mask, "income"] # Interpretation: For all rows in the column "consumption" where the mask is true, set the value to the value in the column "income"

# Print the mean of consumption with and without mask
print("Mean of consumption with mask: ", df_true['consumption'].mean()) # <- With mask
print("Mean of consumption without mask: ", df['consumption'].mean()) # <- Without mask

Mean of consumption with mask:  1.0337728690050054
Mean of consumption without mask:  1.0878444061731818


### 1.5. <a id='toc1_5_'></a>[Dropping observations and columns](#toc0_)

Drop the *ratio* variable and all rows with an income above 1.5. After this, also drop the first 5 rows.

**Why do we call df_true = df_true[mask] instead of just df_true[mask]?**

In [40]:
print(f'before: {df_true.shape[0]} observations, {df_true.shape[1]} variables')
# Our code

# Lets start by dropping the ratio column
df_true = df_true.drop(columns=['ratio'])

# Now we drop all rows with income > 1.5 - same as only keeping rows with income <= 1.5

# Mask for income <= 1.5
mask = df_true['income'] <= 1.5

# Apply the mask to the dataframe
df_true = df_true[mask]

# Drop the first five rows
df_true = df_true.iloc[5:, :] # iloc is used to select rows by index position and loc is used to select rows by label

print(f'after: {df_true.shape[0]} observations, {df_true.shape[1]} variables')

before: 100 observations, 4 variables
after: 65 observations, 3 variables


### 1.6. <a id='toc1_6_'></a>[Renaming](#toc0_)

Rename *consumption* to *cons* and *income* to *inc*.

In [41]:
# Rename the columns to shorter names
df_true = df_true.rename(columns={'income': 'inc', 'consumption': 'cons'})

### 1.7. <a id='toc1_7_'></a>[Income distribution](#toc0_)

Compute the share of income for each decile of the income distribution using the code below as a starting point.

In [42]:
# Find the income deciles
deciles = df_true.quantile([0.1 * i for i in range(1, 10)])

# Create a new column 'decile' that assigns each observation to a decile
df_true['decile'] = pd.qcut(
    df_true['inc'],
    q=10,               # 10 equal-sized groups (deciles)
    labels=range(1, 11) # Label them 1–10
)

# Compute each decile's share of total income
income_share = df_true.groupby('decile', observed=False)['inc'].sum() / df_true['inc'].sum()

# Display the income share of each decile
display(income_share)

decile
1     0.024775
2     0.039289
3     0.061514
4     0.063729
5     0.089852
6     0.098559
7     0.114818
8     0.148799
9     0.149963
10    0.208702
Name: inc, dtype: float64

## 2. <a id='toc2_'></a>[National account identity](#toc0_)

### 2.1. <a id='toc2_1_'></a>[Download](#toc0_)

Consider the following dictionary definitions:

In [43]:
columns_dict = {}
columns_dict['TRANSAKT'] = 'variable'
columns_dict['PRISENHED'] = 'unit'
columns_dict['TID'] = 'year'
columns_dict['INDHOLD'] = 'value'

var_dict = {} # var is for variable
var_dict['P.1 Output'] = 'Y'
var_dict['P.3 Final consumption expenditure'] = 'C'
var_dict['P.3 Government consumption expenditure'] = 'G'
var_dict['P.5 Gross capital formation'] = 'I'
var_dict['P.6 Export of goods and services'] = 'X'
var_dict['P.7 Import of goods and services'] = 'M'

unit_dict = {}
unit_dict['2020-prices, chained values'] = 'real'
unit_dict['Current prices'] = 'nominal'

**Step 1:** Download all of table `nah1`.

In [44]:
# hint, nah1_api = DstApi('?') 
# hint, params = nah1_api._define_base_params(language='en')
# display(params)
# nah1 = nah1_api.get_data(?)

**Step 2:** Rename the columns using `columns_dict` and replace data using `var_dict` and `unit_dict`.

In [45]:
# hint, nah1_true = nah1_true.rename(?)

# for key,value in var_dict.items():
#   nah1.variable.replace(?)

#for key,value in unit_dict.items():
#   nah1.unit.replace(?)

**Step 3:** Only keep rows where the variable is in `[Y, C, G, I, X, M]`. Afterwards convert the `value` column to a float.

In [46]:
# write you code here
# nah1.value = nah1.value.astype('float')

**Step 4:** Discuss what the following summary statistics show.

In [47]:
# nah1.groupby(['variable','unit']).describe()

**Step 5:** Sort the dataset by year

In [48]:
# nah1 = nah1.sort_values(by='?')
# nah1.head()

### 2.2. <a id='toc2_2_'></a>[Merge](#toc0_)

Load population data from Denmark Statistics:

In [49]:
BEFOLK1_api = DstApi('BEFOLK1')
params = BEFOLK1_api._define_base_params(language='en')
display(params)

{'table': 'befolk1',
 'format': 'BULK',
 'lang': 'en',
 'variables': [{'code': 'KØN', 'values': ['*']},
  {'code': 'ALDER', 'values': ['*']},
  {'code': 'CIVILSTAND', 'values': ['*']},
  {'code': 'Tid', 'values': ['*']}]}

In [50]:
for code in ['KØN','CIVILSTAND']:
    print(code)
    display(BEFOLK1_api.variable_levels(code,language='en'))
    print('')

KØN


,id,text
0,TOT,I alt
1,1,Mænd
2,2,Kvinder



CIVILSTAND


,id,text
0,TOT,I alt
1,U,Ugift
2,G,Gift/separeret
3,E,Enke/enkemand
4,F,Fraskilt


In [ ]:
params['variables'][0]['values'] = ['TOT'] 
params['variables'][2]['values'] = ['TOT'] 
BEFOLK1 = BEFOLK1_api.get_data(params=params)
display(BEFOLK1.head())

In [ ]:
BEFOLK1 = BEFOLK1.rename(columns={'TID':'year','INDHOLD':'population'})
BEFOLK1 = BEFOLK1.drop(columns=['KØN','CIVILSTAND'])
pop = BEFOLK1[BEFOLK1.ALDER == 'Age, total'].drop(columns=['ALDER'])
pop.head()

,year,population
0,1999,5313577
101,2018,5781190
202,1972,4975653
303,2008,5475791
404,1973,5007538


**Question a:** Merge the population and the national account data, so there is a new column called `population`. Use the **merge function**.

In [ ]:
# hint, merged = pd.merge(?,?,how='?',on=[?])
# merged_true.tail(10)

**Question b:** Merge the population on again, so there is a new column called `population_alt`. Use the **join method**.

In [ ]:
# pop_with_index = pop.set_index(?)
# pop_with_index = pop_with_index.rename(columns={'population':'population_alt'})
# merged_with_index = merged.set_index(?)
# merged_alt = merged_with_index.join(?)
# merged_alt.tail(10)

**Question c:** Plot GDP per capita and GDP per working-age (18-65) using the code below as a starting point.

In [ ]:
# ages = ?

# working_pop = BEFOLK1[BEFOLK1.ALDER.isin(?)].groupby('year').?
# working_pop = working_pop.drop(columns=['ALDER'])
# working_pop = working_pop.rename(columns={'population':'working_population'})

# merged = pd.merge(nah1, working_pop, how='left', on=['year'])
# merged = pd.merge(merged, pop, how='left', on=['year'])

### 2.3. <a id='toc2_3_'></a>[Split-apply-combine-plot](#toc0_)

Ensure the following code for a **split-apply-combine-plot** can run.

In [ ]:
# # a. split
# nah1_true_grouped = nah1_true.groupby(['variable','unit'])
# nah1_true_grouped_first = nah1_true_grouped.value.first()
# nah1_true_grouped_first.name = 'first'

# # b. apply
# nah1_true = nah1_true.set_index(['variable','unit','year'])
# nah1_true = nah1_true.join(nah1_true_grouped_first,how='left',on=['variable','unit'])
# nah1_true = nah1_true.reset_index()

# # c. combine
# nah1_true['indexed'] = nah1_true['value']/nah1_true['first']

# # d. plot
# def plot(df,variable='indexed'):
#     df_indexed = df.set_index('year')
#     I = df_indexed.unit == 'real'
#     df_indexed[I].groupby(['variable'])[variable].plot(legend=True);
    
# plot(nah1_true)

**Question:** Implement the same split-apply-combine as above using `transform`.

In [ ]:
def first(x): # select the first element in a series
    return x.iloc[0]

# nah1_alt = nah1.copy()
# grouped = nah1_alt.groupby(?)
#nah1_alt[?] = ?.transform(lambda x: ?)
#nah1_alt.head()

In [ ]:
# plot(nah1_alt,variable='index_transform')